# Fine-Tuning the Encoder on a Downstream Target

Self-supervised contrastive pretraining learns *task-agnostic* structure. To
make the embedding **target-discriminative**, attach a classification head and
fine-tune the encoder end-to-end on the labelled target.

Pipeline:
1. Generate synthetic series + a binary target driven by latent risk factors.
2. **Pretrain** a `TSEmbeddingModel` contrastively (VICReg) — task-agnostic.
3. Load the pretrained encoder into a `TSClassifier` (encoder + head).
4. **Two-phase fine-tune**: warm up the head with the encoder frozen, then
   unfreeze and fine-tune everything with **discriminative learning rates**
   (small LR for the pretrained encoder, larger LR for the new head).
5. Compare a frozen-embedding linear probe vs the fine-tuned model.

Single GPU; falls back to CPU for the demo.

In [ ]:
import sys, pathlib
REPO_ROOT = pathlib.Path.cwd()
if not (REPO_ROOT / 'ts_embed').exists():
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT))

import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader

from ts_embed.data import (DatasetPaths, TimeSeriesDataset,
                           TimeFeatureMasker, ContrastiveCollator)
from ts_embed.model import TSEmbeddingModel, TSEncoderConfig, TSClassifier
from ts_embed.loss import VICRegLoss, VICRegConfig

torch.manual_seed(0); np.random.seed(0)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device)

## 1. Synthetic data + a binary target

Three latent factors (`delinq`, `payment`, `balance`) drive feature blocks and
also a binary target. The target *is* recoverable from the series, but a
task-agnostic encoder will not capture it perfectly — that gap is what
fine-tuning closes.

In [ ]:
N, T, F_NUM, F_CAT = 6000, 24, 98, 2
rng = np.random.default_rng(0)

f_delinq  = rng.standard_normal(N).astype(np.float32)
f_payment = rng.standard_normal(N).astype(np.float32)
f_balance = rng.standard_normal(N).astype(np.float32)

numeric = (0.3 * rng.standard_normal((N, T, F_NUM))).astype(np.float32)
t_axis = np.linspace(0, 1, T, dtype=np.float32)[None, :, None]
numeric[:, :, 0:33]  += f_balance[:, None, None] * (0.5 + t_axis)
numeric[:, :, 33:66] += f_payment[:, None, None] + 0.3 * rng.standard_normal(
    (N, T, 33)).astype(np.float32)
numeric[:, :, 66:98] += f_delinq[:, None, None] * t_axis * 1.5

missing = (rng.random((N, T, F_NUM)) < 0.1).astype(np.uint8)
feat_mean = numeric.mean((0, 1), keepdims=True)
numeric = np.where(missing == 1, feat_mean, numeric).astype(np.float32)
categorical = np.stack([
    (rng.random((N, T)) < 0.5).astype(np.int8),
    (rng.random((N, T)) < 0.5).astype(np.int8),
], axis=-1)

# Binary target from the latent factors (+ noise) -> imbalanced, like credit.
logit = 1.3 * f_delinq - 1.0 * f_payment + 0.6 * f_balance - 1.4
prob = 1.0 / (1.0 + np.exp(-logit))
target = (rng.random(N) < prob).astype(np.float32)

data_dir = REPO_ROOT / 'data_demo'; data_dir.mkdir(exist_ok=True)
np.save(data_dir / 'numeric.npy', numeric)
np.save(data_dir / 'missing.npy', missing)
np.save(data_dir / 'categorical.npy', categorical)
np.save(data_dir / 'target.npy', target)
print(f'N={N}  target positive rate = {target.mean():.3f}')

## 2. Contrastive pre-training (task-agnostic)

A short VICReg run to produce a pretrained encoder. In a real project this is
the long run on all 3M accounts; here we keep it brief.

In [ ]:
paths = DatasetPaths(numeric=data_dir / 'numeric.npy',
                     missing=data_dir / 'missing.npy',
                     categorical=data_dir / 'categorical.npy')
pretrain_ds = TimeSeriesDataset(paths)
masker = TimeFeatureMasker(time_mask_prob=0.25, feature_mask_prob=0.30,
                           n_time_spans=2, feat_span_frac=0.5)
pre_loader = DataLoader(pretrain_ds, batch_size=256, shuffle=True, drop_last=True,
                        num_workers=0, collate_fn=ContrastiveCollator(masker))

cfg = TSEncoderConfig(n_numeric=F_NUM, cat_cardinalities=(2, 2), seq_len=T,
                      d_model=128, n_layers=3, n_heads=4, proj_dim=128)
pre_model = TSEmbeddingModel(cfg).to(device)
vicreg = VICRegLoss(VICRegConfig(gather_distributed=False))
pre_opt = torch.optim.AdamW(pre_model.parameters(), lr=1e-3, weight_decay=1e-4)

PRETRAIN_EPOCHS = 6
pre_model.train()
for epoch in range(PRETRAIN_EPOCHS):
    for batch in pre_loader:
        num_a = batch['numeric_a'].to(device); mis_a = batch['missing_a'].to(device)
        keep_a = batch['time_keep_a'].to(device)
        num_b = batch['numeric_b'].to(device); mis_b = batch['missing_b'].to(device)
        keep_b = batch['time_keep_b'].to(device)
        cat_a = batch['categorical_a'].to(device)
        cat_b = batch['categorical_b'].to(device)
        _, z_a = pre_model(num_a, mis_a, cat_a, keep_a)
        _, z_b = pre_model(num_b, mis_b, cat_b, keep_b)
        loss = vicreg(z_a, z_b)['loss']
        pre_opt.zero_grad(set_to_none=True)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(pre_model.parameters(), 1.0)
        pre_opt.step()
    print(f'pretrain epoch {epoch}: vicreg loss = {float(loss):.3f}')

run_dir = REPO_ROOT / 'runs' / 'finetune_demo'; run_dir.mkdir(parents=True, exist_ok=True)
torch.save({'model': pre_model.state_dict(), 'cfg': cfg.__dict__},
           run_dir / 'pretrain.pt')
print('saved pretrained encoder ->', run_dir / 'pretrain.pt')

## 3. Fine-tuning setup: load encoder + classification head

`TSClassifier` = `TSEncoder` + `ClassificationHead`. We load the pretrained
encoder weights, then build a labelled train/val split. The target is
imbalanced, so we pass `pos_weight` to `BCEWithLogitsLoss`.

In [ ]:
# Cast to float for BCEWithLogitsLoss; .float() makes this robust no matter
# which dtype the file was last saved with.
TARGET = torch.from_numpy(np.load(data_dir / 'target.npy')).float()  # (N,)

class IndexedDataset(Dataset):
    def __init__(self, ds): self.ds = ds
    def __len__(self): return len(self.ds)
    def __getitem__(self, i):
        item = self.ds[i]; item['_idx'] = i; return item

def clf_collate(batch):
    idx = torch.tensor([b['_idx'] for b in batch])
    return {
        'numeric': torch.stack([b['numeric'] for b in batch]),
        'missing': torch.stack([b['missing'] for b in batch]),
        'categorical': torch.stack([b['categorical'] for b in batch]),
        'target': TARGET[idx],
    }

full = IndexedDataset(TimeSeriesDataset(paths))
val_n = int(0.2 * len(full)); train_n = len(full) - val_n
train_ds, val_ds = torch.utils.data.random_split(
    full, [train_n, val_n], generator=torch.Generator().manual_seed(0))
train_loader = DataLoader(train_ds, batch_size=256, shuffle=True, drop_last=True,
                          num_workers=0, collate_fn=clf_collate)
val_loader = DataLoader(val_ds, batch_size=512, shuffle=False,
                        num_workers=0, collate_fn=clf_collate)

ckpt = torch.load(run_dir / 'pretrain.pt', map_location='cpu', weights_only=False)
cfg = TSEncoderConfig(**ckpt['cfg'])
clf = TSClassifier(cfg, n_classes=1, head_hidden=128, head_dropout=0.2).to(device)
incompatible = clf.load_pretrained_encoder(ckpt['model'], strict=True)
print('pretrained encoder loaded; incompatible keys:', incompatible)

# Class imbalance -> pos_weight = n_neg / n_pos on the TRAIN split only.
tr_idx = torch.tensor(train_ds.indices)
n_pos = TARGET[tr_idx].sum(); n_neg = len(tr_idx) - n_pos
pos_weight = (n_neg / n_pos.clamp_min(1.0)).to(device)
criterion = torch.nn.BCEWithLogitsLoss(pos_weight=pos_weight)
print(f'train pos={int(n_pos)} neg={int(n_neg)} pos_weight={float(pos_weight):.2f}')

## 4. Two-phase fine-tune

- **Phase 1 (head warm-up)** — encoder frozen and in `eval()` mode (no dropout
  / stable features); only the head trains. This stops large early head
  gradients from wrecking the pretrained encoder weights.
- **Phase 2 (fine-tune)** — unfreeze; train everything with discriminative LRs
  (`encoder_lr` << `head_lr`).

The optimizer holds all parameters from the start; freezing is done via
`requires_grad`, so no optimizer rebuild is needed when we unfreeze.

In [ ]:
def roc_auc(y, s):
    """ROC AUC; sklearn if available, else tie-aware Mann-Whitney."""
    y = np.asarray(y); s = np.asarray(s)
    try:
        from sklearn.metrics import roc_auc_score
        return float(roc_auc_score(y, s))
    except ImportError:
        order = np.argsort(s, kind='mergesort')
        ranks = np.empty(len(s), float)
        sr = s[order]; i = 0
        while i < len(sr):
            j = i
            while j + 1 < len(sr) and sr[j + 1] == sr[i]:
                j += 1
            ranks[order[i:j + 1]] = 0.5 * (i + j) + 1.0  # average rank for ties
            i = j + 1
        pos = y == 1
        npos = int(pos.sum()); nneg = len(y) - npos
        if npos == 0 or nneg == 0:
            return float('nan')
        return float((ranks[pos].sum() - npos * (npos + 1) / 2) / (npos * nneg))

@torch.no_grad()
def evaluate():
    clf.eval()
    ys, ss = [], []
    for b in val_loader:
        logits = clf(b['numeric'].to(device), b['missing'].to(device),
                     b['categorical'].to(device))
        ss.append(torch.sigmoid(logits).float().cpu().numpy())
        ys.append(b['target'].numpy())
    return roc_auc(np.concatenate(ys), np.concatenate(ss))

opt = torch.optim.AdamW(clf.param_groups(encoder_lr=2e-4, head_lr=2e-3,
                                         weight_decay=1e-4))

HEAD_WARMUP, FINETUNE = 3, 10
best_auc, best_state = 0.0, None
for epoch in range(HEAD_WARMUP + FINETUNE):
    finetuning = epoch >= HEAD_WARMUP
    clf.set_encoder_trainable(finetuning)
    clf.train()
    if not finetuning:
        clf.encoder.eval()              # frozen encoder: no dropout
    for b in train_loader:
        logits = clf(b['numeric'].to(device), b['missing'].to(device),
                     b['categorical'].to(device))
        loss = criterion(logits, b['target'].to(device))
        opt.zero_grad(set_to_none=True)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(clf.parameters(), 1.0)
        opt.step()
    auc = evaluate()
    improved = auc > best_auc
    if improved:
        best_auc = auc
        best_state = {k: v.detach().cpu().clone() for k, v in clf.state_dict().items()}
    tag = 'finetune   ' if finetuning else 'head-warmup'
    print(f'epoch {epoch:2d} [{tag}]  val_AUC={auc:.4f}'
          + ('  <- best' if improved else ''))

torch.save({'model': best_state, 'cfg': cfg.__dict__, 'val_auc': best_auc},
           run_dir / 'finetuned_best.pt')
print(f'\nbest val AUC = {best_auc:.4f}  ->  {run_dir / "finetuned_best.pt"}')

## 5. Did fine-tuning help? Frozen linear probe vs fine-tuned

The **linear probe** = logistic regression on the *frozen* pretrained
embedding (what you get if you just dump SSL embeddings into a downstream
model). Fine-tuning should beat it, because the encoder itself moved toward
the target.

In [ ]:
@torch.no_grad()
def embed(loader, encode_fn):
    Z, Y = [], []
    for b in loader:
        z = encode_fn(b['numeric'].to(device), b['missing'].to(device),
                      b['categorical'].to(device))
        Z.append(z.float().cpu().numpy()); Y.append(b['target'].numpy())
    return np.concatenate(Z), np.concatenate(Y)

pre_model.eval()
# train_loader has drop_last=True; use a non-dropping loader for the probe.
probe_loader = DataLoader(train_ds, batch_size=512, shuffle=False,
                          num_workers=0, collate_fn=clf_collate)
Ztr, Ytr = embed(probe_loader, lambda n, m, c: pre_model.encode(n, m, c))
Zva, Yva = embed(val_loader, lambda n, m, c: pre_model.encode(n, m, c))
try:
    from sklearn.linear_model import LogisticRegression
    lp = LogisticRegression(max_iter=2000, class_weight='balanced').fit(Ztr, Ytr)
    probe_auc = roc_auc(Yva, lp.predict_proba(Zva)[:, 1])
except ImportError:
    probe_auc = float('nan')

print(f'frozen pretrained embedding + logistic regression : val AUC = {probe_auc:.4f}')
print(f'fine-tuned encoder + head                          : val AUC = {best_auc:.4f}')
print('\nFine-tuning moves the encoder weights toward the target, so it should')
print('beat the frozen probe -- that gap is the value of fine-tuning.')

## Next steps

- **Real pretraining**: replace step 2 with a full contrastive run (or load a
  checkpoint from `scripts/train_ddp.py`) — `load_pretrained_encoder` accepts
  those checkpoints directly.
- **Regularize fine-tuning**: light masking augmentation on the inputs, layer
  -wise LR decay, or early stopping on `val_AUC` curb overfitting on small
  labelled sets.
- **Multiclass**: set `n_classes > 1`; `forward` then returns `(B, n_classes)`
  logits for `CrossEntropyLoss`.
- **Hybrid features**: concatenate the fine-tuned embedding with your existing
  aggregated features for a downstream GBM, rather than replacing them.